In [1]:
# ============================================================
# CELL 1
# IMPORTS
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os
import json
import random
import time

import cv2
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
from torchvision import transforms
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


Mounted at /content/drive


In [2]:
# ============================================================
# CELL 2
# DATASET SETUP
# ============================================================

DATASET_ROOT = os.path.join(r"c:\Users\priya\Downloads\ok\TinyBayes-ML", "data", "dataset", "tomato")

SAVE_DIR = os.path.join(
    DATASET_ROOT,
    "run"
)

os.makedirs(
    SAVE_DIR,
    exist_ok=True
)

RANDOM_STATE = 42

IMAGE_SIZE = 224

CLASS_NAMES = ['Bacterial_Spot', 'Early_Blight', 'Healthy', 'Late_Blight', 'Leaf_Mold', 'Mosaic_Virus', 'Septoria_Spot', 'Yellow_Virus']

print("="*60)
print("CLASSES")
print("="*60)

print(CLASS_NAMES)

# ------------------------------------------------------------
# Build dataframe from dataset directory
# ------------------------------------------------------------

data = []
for cls_name in CLASS_NAMES:
    cls_dir = os.path.join(DATASET_ROOT, cls_name)
    if os.path.exists(cls_dir):
        for img_file in os.listdir(cls_dir):
            if img_file.lower().endswith(('.jpg', '.jpeg', '.png')):
                data.append({'Image_ID': img_file, 'class': cls_name})

df = pd.DataFrame(data)

print()
print("Total Images :", len(df))
print()
print(df["class"].value_counts())

# ------------------------------------------------------------
# Encode labels
# ------------------------------------------------------------

label_encoder = LabelEncoder()
df["label"] = label_encoder.fit_transform(df["class"])
NUM_CLASSES = len(label_encoder.classes_)

print()
print("Label Mapping")
for i, cls in enumerate(label_encoder.classes_):
    print(i, "->", cls)

# ------------------------------------------------------------
# Train / Validation Split (Stratified 80 / 20)
# ------------------------------------------------------------

df_train, df_val = train_test_split(
    df,
    test_size=0.20,
    stratify=df["label"],
    random_state=RANDOM_STATE
)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)

print()
print("="*60)
print("TRAIN SPLIT")
print("="*60)
print(df_train["class"].value_counts())

print()
print("="*60)
print("VALIDATION SPLIT")
print("="*60)
print(df_val["class"].value_counts())

# ------------------------------------------------------------
# Save splits
# ------------------------------------------------------------

df_train.to_csv(os.path.join(SAVE_DIR, "train_split.csv"), index=False)
df_val.to_csv(os.path.join(SAVE_DIR, "validation_split.csv"), index=False)

print()
print("Saved")
print(os.path.join(SAVE_DIR, "train_split.csv"))
print(os.path.join(SAVE_DIR, "validation_split.csv"))


CLASSES
['Bacterial_Spot', 'Early_Blight', 'Healthy', 'Late_Blight', 'Leaf_Mold', 'Mosaic_Virus', 'Septoria_Spot', 'Yellow_Virus']

Total Images : 2662

class
Septoria_Spot     387
Late_Blight       351
Bacterial_Spot    348
Leaf_Mold         335
Early_Blight      324
Yellow_Virus      319
Healthy           304
Mosaic_Virus      294

Label Mapping
0 -> Bacterial_Spot
1 -> Early_Blight
2 -> Healthy
3 -> Late_Blight
4 -> Leaf_Mold
5 -> Mosaic_Virus
6 -> Septoria_Spot
7 -> Yellow_Virus

TRAIN SPLIT
class
Septoria_Spot     310
Late_Blight       281
Bacterial_Spot    278
Leaf_Mold         268
Early_Blight      259
Yellow_Virus      255
Healthy           243
Mosaic_Virus      235

VALIDATION SPLIT
class
Septoria_Spot     77
Bacterial_Spot    70
Late_Blight       70
Leaf_Mold         67
Early_Blight      65
Yellow_Virus      64
Healthy           61
Mosaic_Virus      59

Saved
c:\Users\priya\Downloads\ok\TinyBayes-ML\data\dataset\tomato\run\train_split.csv
c:\Users\priya\Downloads\ok\TinyBayes

In [3]:
train_ids = set(df_train["Image_ID"])
val_ids = set(df_val["Image_ID"])

overlap = train_ids.intersection(val_ids)

print("Overlap:", len(overlap))


Overlap: 0


In [4]:
# ============================================================
# SAVE CLASS NAMES
# ============================================================

CLASS_NAMES_PATH = os.path.join(
    SAVE_DIR,
    "class_names.json"
)

with open(CLASS_NAMES_PATH, "w") as f:
    json.dump(CLASS_NAMES, f, indent=4)

print("Class names saved to:")
print(CLASS_NAMES_PATH)

print()
print("Classes:")
print(CLASS_NAMES)


Class names saved to:
c:\Users\priya\Downloads\ok\TinyBayes-ML\data\dataset\tomato\run\class_names.json

Classes:
['Bacterial_Spot', 'Early_Blight', 'Healthy', 'Late_Blight', 'Leaf_Mold', 'Mosaic_Virus', 'Septoria_Spot', 'Yellow_Virus']


In [5]:
# ============================================================
# CELL 3
# EXTRACT MOBILENET FEATURES
# ============================================================

TRAIN_FEATURE_PATH = os.path.join(
    SAVE_DIR,
    "train_features.csv"
)

VAL_FEATURE_PATH = os.path.join(
    SAVE_DIR,
    "validation_features.csv"
)

if os.path.exists(TRAIN_FEATURE_PATH) and os.path.exists(VAL_FEATURE_PATH):

    print("="*60)
    print("FEATURE FILES FOUND")
    print("="*60)

    train_features_df = pd.read_csv(TRAIN_FEATURE_PATH)
    val_features_df = pd.read_csv(VAL_FEATURE_PATH)

    print(train_features_df.shape)
    print(val_features_df.shape)
    print()
    print(train_features_df.head())


FEATURE FILES FOUND
(2129, 578)
(533, 578)

                              Image_ID           class    feat_0    feat_1    feat_2
0  Tomato Septoria leaf spot (109).jpg   Septoria_Spot  0.121746 -0.007444  0.558986
1                 Tomato leaf (48).jpg         Healthy -0.179262 -0.030738  0.048753
2  Tomato leaf bacterial spot (13).jpg  Bacterial_Spot  0.208197  0.149698  0.186010
3  Tomato Septoria leaf spot (101).jpg   Septoria_Spot -0.118110 -0.086302  0.773709
4     Tomato leaf late blight (85).jpg     Late_Blight -0.165592 -0.043943  0.086602

Train features shape: (2129, 578)
Val features shape:   (533, 578)


In [6]:
# ============================================================
# CELL 4
# TRAIN JACOBI-DMR
# ============================================================

import time

time_jacobi_start = time.time()

# ------------------------------------------------------------
# Separate features and labels
# ------------------------------------------------------------

feat_cols = [c for c in train_features_df.columns if c.startswith("feat_")]

X_train = train_features_df[feat_cols].values.astype(np.float32)
y_train = train_features_df["class"].values

X_val = val_features_df[feat_cols].values.astype(np.float32)
y_val = val_features_df["class"].values

# ------------------------------------------------------------
# One-hot encode targets
# ------------------------------------------------------------

y_train_df = pd.DataFrame({"c": y_train})
y_one_hot = pd.get_dummies(y_train_df["c"])

for cls in CLASS_NAMES:
    if cls not in y_one_hot.columns:
        y_one_hot[cls] = 0

y_one_hot = y_one_hot[CLASS_NAMES]

# ------------------------------------------------------------
# Jacobi regularized solve
# ------------------------------------------------------------

N = len(X_train)
a = 1.0 / N
b = 1.0 / N
k = 1.0

identity = np.eye(X_train.shape[1], dtype=np.float32)
regularization_lambda = 1.0

XtX = np.matmul(X_train.T, X_train)
XtX_reg = XtX + regularization_lambda * identity

jacobi_coefficients = {}

for cls in CLASS_NAMES:
    y_c = y_one_hot[cls].values
    eta = np.log((y_c + a) / (1.0 + k * b))
    Xty = np.matmul(X_train.T, eta)
    beta = np.linalg.solve(XtX_reg, Xty)
    jacobi_coefficients[cls] = beta

# ------------------------------------------------------------
# Validation inference
# ------------------------------------------------------------

preds_jacobi = []
for i in range(len(X_val)):
    x = X_val[i]
    scores = {cls: float(np.dot(x, jacobi_coefficients[cls])) for cls in CLASS_NAMES}
    preds_jacobi.append(max(scores, key=scores.get))

time_jacobi = time.time() - time_jacobi_start
acc_jacobi = accuracy_score(y_val, preds_jacobi)

print()
print("="*60)
print("JACOBI-DMR")
print("="*60)
print(f"Accuracy: {acc_jacobi:.4f}")
print(f"Time: {time_jacobi:.4f}s")
print()
print(classification_report(y_val, preds_jacobi, labels=CLASS_NAMES))
print()
print("Confusion Matrix:")
print()
print(confusion_matrix(y_val, preds_jacobi, labels=CLASS_NAMES))

# ------------------------------------------------------------
# Save coefficients
# ------------------------------------------------------------

JACOBI_JSON_PATH = os.path.join(SAVE_DIR, "jacobi_coefficients.json")

save_dict = {cls: [float(v) for v in jacobi_coefficients[cls]] for cls in CLASS_NAMES}

with open(JACOBI_JSON_PATH, "w") as f:
    json.dump(save_dict, f, indent=4)

print()
print("Jacobi coefficients saved to:")
print(JACOBI_JSON_PATH)



JACOBI-DMR
Accuracy: 0.7880
Time: 3.2882s

                precision    recall  f1-score   support

Bacterial_Spot       0.72      0.71      0.72        70
  Early_Blight       0.76      0.63      0.69        65
       Healthy       0.86      0.90      0.88        61
   Late_Blight       0.73      0.74      0.74        70
     Leaf_Mold       0.79      0.82      0.80        67
  Mosaic_Virus       0.89      0.85      0.87        59
 Septoria_Spot       0.70      0.75      0.72        77
  Yellow_Virus       0.89      0.92      0.91        64

      accuracy                           0.79       533
     macro avg       0.79      0.79      0.79       533
  weighted avg       0.79      0.79      0.79       533


Confusion Matrix:

[[50  1  3  4  2  0  8  2]
 [ 5 41  3  5  1  0 10  0]
 [ 1  0 55  1  2  0  1  1]
 [ 1  6  0 52  5  3  2  1]
 [ 0  1  1  7 55  1  2  0]
 [ 3  0  0  0  2 50  2  2]
 [ 9  4  1  2  1  1 58  1]
 [ 0  1  1  0  2  1  0 59]]

Jacobi coefficients saved to:
c:\Users\priy

In [7]:
# ============================================================
# RANDOM FOREST
# ============================================================

from sklearn.ensemble import RandomForestClassifier

import time

time_rf_start = time.time()

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_model.fit(X_train, y_train)

preds_rf = rf_model.predict(X_val)

time_rf = time.time() - time_rf_start

acc_rf = accuracy_score(y_val, preds_rf)

print()
print("="*60)
print("RANDOM FOREST")
print("="*60)

print(f"Accuracy: {acc_rf:.4f}")
print(f"Time: {time_rf:.4f}s")
print()

print(classification_report(y_val, preds_rf, labels=CLASS_NAMES))
print()

print("Confusion Matrix:")
print()
print(confusion_matrix(y_val, preds_rf, labels=CLASS_NAMES))



RANDOM FOREST
Accuracy: 0.7749
Time: 8.5285s

                precision    recall  f1-score   support

Bacterial_Spot       0.72      0.69      0.70        70
  Early_Blight       0.68      0.62      0.65        65
       Healthy       0.87      0.90      0.89        61
   Late_Blight       0.78      0.80      0.79        70
     Leaf_Mold       0.86      0.72      0.78        67
  Mosaic_Virus       0.93      0.85      0.88        59
 Septoria_Spot       0.63      0.78      0.70        77
  Yellow_Virus       0.84      0.88      0.85        64

      accuracy                           0.77       533
     macro avg       0.79      0.78      0.78       533
  weighted avg       0.78      0.77      0.78       533


Confusion Matrix:

[[48  4  0  5  1  0  8  4]
 [ 6 40  4  5  0  0  9  1]
 [ 0  1 55  0  2  0  3  0]
 [ 0 11  0 56  0  0  3  0]
 [ 2  0  3  6 48  0  6  2]
 [ 2  1  0  0  1 50  3  2]
 [ 9  2  1  0  3  0 60  2]
 [ 0  0  0  0  1  4  3 56]]


In [8]:
# ============================================================
# SUPPORT VECTOR MACHINE (SVM)
# ============================================================

from sklearn.svm import SVC

import time

time_svm_start = time.time()

svm_model = SVC(
    kernel='rbf',
    C=1.0,
    gamma='scale',
    random_state=42
)

svm_model.fit(X_train, y_train)

preds_svm = svm_model.predict(X_val)

time_svm = time.time() - time_svm_start

acc_svm = accuracy_score(y_val, preds_svm)

print()
print("="*60)
print("SUPPORT VECTOR MACHINE")
print("="*60)

print(f"Accuracy: {acc_svm:.4f}")
print(f"Time: {time_svm:.4f}s")
print()

print(classification_report(y_val, preds_svm, labels=CLASS_NAMES))
print()

print("Confusion Matrix:")
print()
print(confusion_matrix(y_val, preds_svm, labels=CLASS_NAMES))



SUPPORT VECTOR MACHINE
Accuracy: 0.8199
Time: 3.1676s

                precision    recall  f1-score   support

Bacterial_Spot       0.74      0.73      0.73        70
  Early_Blight       0.80      0.69      0.74        65
       Healthy       0.95      0.89      0.92        61
   Late_Blight       0.78      0.81      0.80        70
     Leaf_Mold       0.88      0.91      0.90        67
  Mosaic_Virus       0.94      0.85      0.89        59
 Septoria_Spot       0.70      0.77      0.73        77
  Yellow_Virus       0.83      0.94      0.88        64

      accuracy                           0.82       533
     macro avg       0.83      0.82      0.82       533
  weighted avg       0.82      0.82      0.82       533


Confusion Matrix:

[[51  1  0  5  1  1  7  4]
 [ 3 45  2  6  0  0  8  1]
 [ 0  0 54  1  2  0  2  2]
 [ 2  8  0 57  1  0  2  0]
 [ 3  0  0  1 61  1  1  0]
 [ 1  0  0  2  1 50  3  2]
 [ 9  2  1  1  2  0 59  3]
 [ 0  0  0  0  1  1  2 60]]


In [9]:
# ============================================================
# RIDGE CLASSIFIER
# ============================================================

from sklearn.linear_model import RidgeClassifier

import time

time_ridge_start = time.time()

ridge_model = RidgeClassifier(
    alpha=1.0,
    random_state=42
)

ridge_model.fit(X_train, y_train)

preds_ridge = ridge_model.predict(X_val)

time_ridge = time.time() - time_ridge_start

acc_ridge = accuracy_score(y_val, preds_ridge)

print()
print("="*60)
print("RIDGE CLASSIFIER")
print("="*60)

print(f"Accuracy: {acc_ridge:.4f}")
print(f"Time: {time_ridge:.4f}s")
print()

print(classification_report(y_val, preds_ridge, labels=CLASS_NAMES))
print()

print("Confusion Matrix:")
print()
print(confusion_matrix(y_val, preds_ridge, labels=CLASS_NAMES))



RIDGE CLASSIFIER
Accuracy: 0.7880
Time: 0.0977s

                precision    recall  f1-score   support

Bacterial_Spot       0.72      0.70      0.71        70
  Early_Blight       0.76      0.63      0.69        65
       Healthy       0.85      0.90      0.87        61
   Late_Blight       0.74      0.74      0.74        70
     Leaf_Mold       0.79      0.84      0.81        67
  Mosaic_Virus       0.89      0.85      0.87        59
 Septoria_Spot       0.69      0.75      0.72        77
  Yellow_Virus       0.91      0.92      0.91        64

      accuracy                           0.79       533
     macro avg       0.79      0.79      0.79       533
  weighted avg       0.79      0.79      0.79       533


Confusion Matrix:

[[49  1  3  4  3  0  9  1]
 [ 5 41  3  5  1  0 10  0]
 [ 1  0 55  1  2  0  1  1]
 [ 1  6  0 52  5  3  2  1]
 [ 0  1  1  6 56  1  2  0]
 [ 3  0  0  0  2 50  2  2]
 [ 9  4  2  2  0  1 58  1]
 [ 0  1  1  0  2  1  0 59]]


In [10]:
# ============================================================
# LOGISTIC REGRESSION (L1 / LASSO)
# ============================================================

from sklearn.linear_model import LogisticRegression

import time

time_lr_start = time.time()

lr_model = LogisticRegression(
    C=1.0,
    max_iter=1000,
    random_state=42
)

lr_model.fit(X_train, y_train)

preds_lr = lr_model.predict(X_val)

time_lr = time.time() - time_lr_start

acc_lr = accuracy_score(y_val, preds_lr)

print()
print("="*60)
print("LOGISTIC REGRESSION (L1 / LASSO)")
print("="*60)

print(f"Accuracy: {acc_lr:.4f}")
print(f"Time: {time_lr:.4f}s")
print()

print(classification_report(y_val, preds_lr, labels=CLASS_NAMES))
print()

print("Confusion Matrix:")
print()
print(confusion_matrix(y_val, preds_lr, labels=CLASS_NAMES))



LOGISTIC REGRESSION (L1 / LASSO)
Accuracy: 0.8124
Time: 10.4738s

                precision    recall  f1-score   support

Bacterial_Spot       0.80      0.70      0.75        70
  Early_Blight       0.70      0.75      0.73        65
       Healthy       0.95      0.92      0.93        61
   Late_Blight       0.76      0.76      0.76        70
     Leaf_Mold       0.83      0.90      0.86        67
  Mosaic_Virus       0.89      0.86      0.88        59
 Septoria_Spot       0.70      0.75      0.72        77
  Yellow_Virus       0.93      0.89      0.91        64

      accuracy                           0.81       533
     macro avg       0.82      0.82      0.82       533
  weighted avg       0.82      0.81      0.81       533


Confusion Matrix:

[[49  3  1  4  2  1  8  2]
 [ 2 49  1  5  1  0  7  0]
 [ 1  0 56  0  3  0  1  0]
 [ 0 12  1 53  1  0  3  0]
 [ 0  1  0  3 60  1  2  0]
 [ 2  1  0  0  2 51  3  0]
 [ 7  4  0  4  1  1 58  2]
 [ 0  0  0  1  2  3  1 57]]


In [11]:
# ============================================================
# FINAL MODEL COMPARISON
# ============================================================

comparison_df = pd.DataFrame({
    "Model": [
        "Support Vector Machine",
        "Ridge Classifier",
        "Logistic Regression (L1)",
        "Jacobi-DMR",
        "Random Forest"
    ],
    "Accuracy": [
        acc_svm,
        acc_ridge,
        acc_lr,
        acc_jacobi,
        acc_rf
    ],
    "Execution Time (s)": [
        time_svm,
        time_ridge,
        time_lr,
        time_jacobi,
        time_rf
    ]
})

comparison_df = comparison_df.sort_values(
    by="Accuracy",
    ascending=False
).reset_index(drop=True)

print()
print("="*70)
print("MODEL COMPARISON")
print("="*70)

print(comparison_df.to_string(index=False))

comparison_path = os.path.join(
    SAVE_DIR,
    "model_comparison.csv"
)

comparison_df.to_csv(
    comparison_path,
    index=False
)

print()
print(f"Comparison saved to:\n{comparison_path}")



MODEL COMPARISON
                   Model  Accuracy  Execution Time (s)
  Support Vector Machine    0.8199              3.1676
Logistic Regression (L1)    0.8124             10.4738
        Ridge Classifier    0.7880              0.0977
              Jacobi-DMR    0.7880              3.2882
           Random Forest    0.7749              8.5285

Comparison saved to:
c:\Users\priya\Downloads\ok\TinyBayes-ML\data\dataset\tomato\run\model_comparison.csv
